In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from konlpy.tag import Mecab
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten, Embedding
from tensorflow.keras.utils import to_categorical

2025-05-22 22:50:56.987678: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-22 22:50:57.001238: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747921857.015422   23900 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747921857.021105   23900 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747921857.035770   23900 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
test_data = pd.read_csv("./data/bank_app_reviews_test.csv")
test_data.head(2)

,리뷰일,평점,사용자리뷰,업체답변,은행명
0,2024-02-08,5,고경민계장님감사해요,"안녕하세요 최순녀 고객님. 칭찬 진심으로 감사드리며, 더욱 편리하고 안정적인 서비스...",우리
1,2023-07-24,5,저축목표피드 새로 생긴거 너무좋은데 분명 카테고리를 저축으로 했는데 왜 인식이 안되...,"신아​ 님, 안녕하세요? 뱅크샐러드 고객감동팀​입니다. 소중한 시간내어 고객센터에 ...",뱅크샐러드


In [3]:
import re

def clean_text(text):
    cleaned = re.sub(r'[^가-힣a-zA-Z0-9\s]','', text) #한글, 영문, 숫자
    cleaned = re.sub(r'\s+', ' ', cleaned) # 연속된 공백을 하나의 공백
    return cleaned.strip()

In [4]:
test_data['사용자리뷰'] = test_data['사용자리뷰'].apply(clean_text)
test_data['사용자리뷰']

0                                              고경민계장님감사해요
1       저축목표피드 새로 생긴거 너무좋은데 분명 카테고리를 저축으로 했는데 왜 인식이 안되...
2         아니 이딴걸 편리하게 사용하는앱이라고 쳐만들엇나 이렇게 불편하게만든건 일부러그런거에요
3       몇 년째 만족하며 사용중이라 조금식 개선되어거는 모습에 만족하며 사용중입니다 하지만...
4                                스타뱅킹을 사용 하고나서부터 편안해서 좋아요
                              ...                        
9529    만보기 이벤트는 실망스러워요 후기 말투 다 똑같고 사기 맞죠 양심이 참 정직하게 확...
9530                   기능이 많아 다 사용해보진 못 했지만 대체적으로 편한거 같아요
9531                                                편리하네요
9532                                            사용하기 편리해요
9533    너무 후져서 오랜만에 어플이용하다가 욕했답니다 인증방식이 2010년대에 머물러계심 ...
Name: 사용자리뷰, Length: 9534, dtype: object

In [5]:
test_data['is_good'] = test_data['평점'].apply(lambda x: 1 if x >=4 else 0)
test_data['is_good']

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

In [6]:
mecab = Mecab()

In [7]:
tokenized_docs = test_data['사용자리뷰'].apply(mecab.morphs)

In [8]:
tokenized_docs[0]

['고경민', '계장', '님', '감사', '해요']

# train에서 사용했던 tokenizer를 불러와서 one hot encoding

In [9]:
import joblib

In [10]:
token = joblib.load("./model/bank_app_tokenizer.joblib")

In [11]:
x = token.texts_to_sequences(tokenized_docs)
print(x[0])

[6248, 327, 111, 71]


# train에서 사용했던 패딩 길이(모델에 넣을 컬럼 수)

In [12]:
max_length = joblib.load("./model/bank_app_max_length.joblib")
print(max_length)

302


In [13]:
X_padded = pad_sequences(x, maxlen=max_length, padding='post')
print(X_padded[1])

[ 216  717  370 1524   39   44    8  148 1025  724   27 1033   31   43
   14   51  117    3    6    9    4  861  117    9  414  224  112    9
  164  409 2261 1336   20    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 

In [14]:
len(X_padded[1])

302

In [15]:
y = test_data['is_good']
y

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

# 모델 불러와서 예측하고 결과 비교하기

In [16]:
birnn_best = load_model("./model/bank_app_review_birnn.keras")
cnn_lstm_best = load_model("./model/bank_app_review_lstm_cnn.keras")
attn_best = load_model("./model/bank_app_review_attn_model.keras")

I0000 00:00:1747921864.265785   23900 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 560 MB memory:  -> device: 0, name: NVIDIA GeForce MX450, pci bus id: 0000:01:00.0, compute capability: 7.5


In [17]:
birnn_pred = birnn_best.predict(X_padded)
cnn_latm_pred = cnn_lstm_best.predict(X_padded)
attn_pred = attn_best.predict(X_padded)

I0000 00:00:1747921872.292163   27883 service.cc:152] XLA service 0x557a891495a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1747921872.292243   27883 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce MX450, Compute Capability 7.5
2025-05-22 22:51:12.332152: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1747921872.411295   27883 cuda_dnn.cc:529] Loaded cuDNN version 90300


  4/298 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step

I0000 00:00:1747921873.417762   27883 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


298/298 ━━━━━━━━━━━━━━━━━━━━ 17s 51ms/step
298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step
298/298 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step


In [18]:
birnn_pred = pd.DataFrame(birnn_pred)
cnn_lstm_pred = pd.DataFrame(cnn_latm_pred)
attn_pred = pd.DataFrame(attn_pred)

In [19]:
y

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

In [20]:
y = pd.DataFrame(y)

In [21]:
birnn_result = y.join(birnn_pred)
cnn_lstm_result = y.join(cnn_lstm_pred)
attn_result = y.join(attn_pred)

In [22]:
birnn_result.loc[:, 0] = birnn_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)
cnn_lstm_result.loc[:, 0] = cnn_lstm_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)
attn_result.loc[:, 0] = attn_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)

In [23]:
birnn_result

,is_good,0
0,1,1.0
1,1,0.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,1.0
9531,1,1.0
9532,1,1.0


In [24]:
cnn_lstm_result

,is_good,0
0,1,1.0
1,1,0.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,0.0
9531,1,1.0
9532,1,1.0


In [25]:
attn_result

,is_good,0
0,1,1.0
1,1,0.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,1.0
9531,1,1.0
9532,1,1.0


In [26]:
from sklearn.metrics import classification_report

In [27]:
print(classification_report(birnn_result['is_good'], birnn_result[0]))

              precision    recall  f1-score   support

           0       0.87      0.83      0.85      3862
           1       0.89      0.92      0.90      5672

    accuracy                           0.88      9534
   macro avg       0.88      0.87      0.88      9534
weighted avg       0.88      0.88      0.88      9534



In [28]:
print(classification_report(cnn_lstm_result['is_good'], cnn_lstm_result[0]))

              precision    recall  f1-score   support

           0       0.85      0.92      0.88      3862
           1       0.94      0.89      0.91      5672

    accuracy                           0.90      9534
   macro avg       0.89      0.90      0.90      9534
weighted avg       0.90      0.90      0.90      9534



In [29]:
print(classification_report(attn_result['is_good'], attn_result[0]))

              precision    recall  f1-score   support

           0       0.87      0.89      0.88      3862
           1       0.93      0.91      0.92      5672

    accuracy                           0.90      9534
   macro avg       0.90      0.90      0.90      9534
weighted avg       0.90      0.90      0.90      9534



# evaluate

In [30]:
%%time
birnn_best.evaluate(X_padded, test_data['is_good'])

298/298 ━━━━━━━━━━━━━━━━━━━━ 19s 56ms/step - accuracy: 0.8863 - auc: 0.9438 - loss: 0.3014
CPU times: user 11.5 s, sys: 5.21 s, total: 16.7 s
Wall time: 19.3 s


[0.302606999874115, 0.8820012807846069, 0.9432716369628906]

In [31]:
%%time
cnn_lstm_best.evaluate(X_padded, test_data['is_good'])

298/298 ━━━━━━━━━━━━━━━━━━━━ 16s 45ms/step - accuracy: 0.9023 - auc: 0.9574 - loss: 0.2552
CPU times: user 13 s, sys: 5.66 s, total: 18.7 s
Wall time: 16.3 s


[0.2570849657058716, 0.8993077278137207, 0.9570852518081665]

In [32]:
%%time
attn_best.evaluate(X_padded, test_data['is_good'])

298/298 ━━━━━━━━━━━━━━━━━━━━ 37s 121ms/step - accuracy: 0.9030 - auc: 0.9575 - loss: 0.2550
CPU times: user 20.8 s, sys: 10.7 s, total: 31.5 s
Wall time: 37.3 s


[0.2612304985523224, 0.9010908603668213, 0.9557893872261047]